# 03 · Managed Agents: las mismas palancas, el loop corre en Anthropic
Modelo `claude-sonnet-5`. Cada celda crea un agente (configuración versionada) y abre una sesión. Pregunta fija: *¿Cómo cambió el margen por proyecto de junio a julio, y por qué?*.
Requiere `ANTHROPIC_API_KEY` y `MANAGED_ENV_ID`.

In [1]:
import os, json, logging, warnings
from pathlib import Path

logging.getLogger("anthropic").setLevel(logging.ERROR); warnings.filterwarnings("ignore")

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))      # lee notebooks/.env si existe; no pisa variables ya exportadas

if "ANTHROPIC_API_KEY" not in os.environ:
    raise SystemExit("Falta ANTHROPIC_API_KEY en el entorno.")

MODEL = "claude-sonnet-5"
WS = (Path.cwd() if Path.cwd().name == "workspace" else Path("workspace")).resolve()  # contabilidad.csv (sintético) + .claude/skills/
(WS / "CLAUDE.md").unlink(missing_ok=True) # cada corrida empieza sin contexto de proyecto
PREGUNTA = '¿Cómo cambió el margen por proyecto de junio a julio, y por qué?'
SYSTEM = """Eres un analista financiero. Trabajas en el directorio actual, donde está ./contabilidad.csv
(separador ';', decimal con coma; columnas Period, AccountId, Debito, Credito, ProjectId, CostCenterName). No busques en otros directorios.
Reglas: ingreso = cuentas que empiezan por 4 (Credito - Debito); costo = cuentas 6 y 7 (Debito - Credito).
Margen = (ingreso - costo) / ingreso. Period 6 = junio, 7 = julio.
Verifica antes de responder. Salida: cifra por proyecto, el cálculo y la causa del cambio. Español, corto."""
SYSTEM_CORTO = 'Eres un analista financiero. Español, corto.'

import anthropic
from anthropic.lib import files_from_dir

if "MANAGED_ENV_ID" not in os.environ:
    raise SystemExit("Falta MANAGED_ENV_ID (Console → Managed Agents → Environments).")
ENV_ID = os.environ["MANAGED_ENV_ID"]
MOUNT = "/mnt/session/uploads/contabilidad.csv"
client = anthropic.Anthropic(max_retries=5)
creados = []   # agentes creados aquí; se archivan al final

def agente(nombre, **cfg):
    a = client.beta.agents.create(name=f"nb03 · {nombre}", model=MODEL, **cfg)
    creados.append(a.id)
    return a

def correr(agent, pregunta=PREGUNTA, resources=None, herramientas=None):
    """Abre una sesión, imprime los eventos y resuelve custom tools (herramientas = {nombre: función})."""
    s = client.beta.sessions.create(agent=agent.id, environment_id=ENV_ID, resources=resources or [],
        initial_events=[{"type": "user.message", "content": [{"type": "text", "text": pregunta}]}])
    print(f"sesión {s.id}")
    eventos = {}
    with client.beta.sessions.events.stream(s.id) as stream:
        for ev in stream:
            eventos[ev.id] = ev
            if ev.type == "agent.message":
                print("".join(b.text for b in ev.content if getattr(b, "type", "") == "text"))
            elif ev.type == "agent.tool_use":
                print(f"→ {ev.name}: {str(ev.input)[:90]}")
            elif ev.type == "agent.custom_tool_use":
                print(f"→ custom {ev.name}: {ev.input}")
            elif ev.type == "session.status_idle":
                sr = ev.stop_reason
                if sr and sr.type == "requires_action":
                    for eid in sr.event_ids:
                        te = eventos[eid]
                        salida = herramientas[te.name](**te.input)
                        client.beta.sessions.events.send(s.id, events=[{"type": "user.custom_tool_result",
                            "custom_tool_use_id": eid, "content": [{"type": "text", "text": salida}]}])
                else:
                    break
            elif ev.type == "session.error":
                raise RuntimeError(ev)
    print(f"\nhttps://platform.claude.com/workspaces/default/sessions/{s.id}")
    return s

## 1 · Solo el modelo
`tools=[]`: sandbox sin herramientas, sin archivo.

In [2]:
a1 = agente("solo modelo", tools=[])
correr(a1);

sesión sesn_0122gYEYXPnMk2UPiwLLWNRe


No tengo acceso a datos sobre los márgenes de tus proyectos en esta conversación. Para poder analizar el cambio entre junio y julio, necesito que me proporciones la información, por ejemplo:

- Un archivo (Excel, CSV, PDF) con los datos financieros por proyecto de ambos meses
- Acceso a una base de datos o export desde tu sistema (ERP, contabilidad, etc.)
- O bien, si ya tienes las cifras, puedes compartirlas directamente en el chat (ingresos, costos, y margen por proyecto para junio y julio)

Con esos datos podría:

1. Calcular el margen (%) y margen absoluto por proyecto en cada mes
2. Identificar qué proyectos subieron o bajaron de margen y en cuánto
3. Desglosar las causas probables (cambios en costos, en precios/ingresos, en volumen, mezcla de proyectos, etc.)
4. Señalar los proyectos que más explican la variación total

¿Puedes compartir el archivo o los datos correspondientes?

https://platform.claude.com/workspaces/default/sessions/sesn_0122gYEYXPnMk2UPiwLLWNRe


## 2 · + system prompt

In [3]:
a2 = agente("system prompt", system=SYSTEM, tools=[])
correr(a2);

sesión sesn_017gdpaMBSxeQ7e5KaAudM9n


Voy a analizar el archivo directamente.

read_file readFile
{
"path": "./contabilidad.csv"
}

https://platform.claude.com/workspaces/default/sessions/sesn_017gdpaMBSxeQ7e5KaAudM9n


## 3 · + bash y el archivo montado
Del toolset solo `bash` y `read` (un archivo montado exige `read`). El CSV se sube con Files API y se monta en la sesión.

In [4]:
archivo = client.beta.files.upload(file=WS / "contabilidad.csv")
CSV_MONTADO = [{"type": "file", "file_id": archivo.id, "mount_path": MOUNT}]
SYSTEM_MA = SYSTEM.replace("en el directorio actual, donde está ./contabilidad.csv", f"con el archivo {MOUNT}")

a3 = agente("bash", system=SYSTEM_MA,
            tools=[{"type": "agent_toolset_20260401", "default_config": {"enabled": False},
                    "configs": [{"name": "bash", "enabled": True}, {"name": "read", "enabled": True}]}])
correr(a3, resources=CSV_MONTADO);

sesión sesn_01JBdkgJzSZWMd4CaucWpJW6


→ bash: {'command': 'cd /mnt/session/uploads && ls -la && head -20 contabilidad.csv'}


→ bash: {'command': "cd /mnt/session/uploads && python3 << 'EOF'\nimport csv\nfrom collections imp


→ bash: {'command': 'cd /mnt/session/uploads && python3 << \'EOF\'\nimport csv\nfrom collections i


→ bash: {'command': 'cd /mnt/session/uploads && python3 << \'EOF\'\nimport csv\nfrom collections i


## Verificación de datos
Datos tomados de `contabilidad.csv`, filtrando Period=6 (jun) y 7 (jul). Ingreso = Créditos−Débitos en cuentas que inician en 4; Costo = Débitos−Créditos en cuentas 6 y 7. Margen = (Ingreso−Costo)/Ingreso.

Nota: Proyectos "0" y "5001" (Gerencias) no tienen ingresos asociados (son cuentas de costo/gasto administrativo), por lo que no aplican margen.

## Margen por proyecto (jun → jul)

| Proyecto (Centro de costo) | Margen Jun | Margen Jul | Cambio |
|---|---|---|---|
| 1030 – WAREHOUSE | 30,6% | 33,0% | +2,4 pp ▲ |
| 1045 – WAREHOUSE | 33,7% | 11,5% | **-22,2 pp ▼** |
| 2210 – FIRST MILE | 38,0% | 48,0% | +10,0 pp ▲ |
| 2235 – FIRST MILE | 30,0% | 22,0% | -8,0 pp ▼ |
| 3310 – LAST MILE COL. | 22,7% | 19,5% | -3,2 pp ▼ |
| 3322 – LAST MILE COL. | 30,0% | 29,7% | -0,3 pp ≈ |
| 4410 – LONG HAUL | 30,1% | 28,0% | -2,1 pp ▼ |

## Cálculo (ejemplo 1045)
Jun: Ingreso $222.712.500 − Costo $147.650.705 = $75.061.795 → margen 33,7%
Jul: Ingreso $230.000.000 − Costo $203


https://platform.claude.com/workspaces/default/sessions/sesn_01JBdkgJzSZWMd4CaucWpJW6


## 4 · + toolset completo + skill
La skill se sube al workspace (`client.skills.create`) y se adjunta al agente. System prompt corto.

In [5]:
skill = client.skills.create(files=files_from_dir(str(WS / ".claude/skills/margen-por-proyecto")))
print("skill", skill.id)
SYSTEM_CORTO = f"Eres un analista financiero. El archivo está en {MOUNT}. Español, corto."

a4 = agente("toolset + skill", system=SYSTEM_CORTO,
            tools=[{"type": "agent_toolset_20260401"}],
            skills=[{"type": "custom", "skill_id": skill.id, "version": "latest"}])
correr(a4, resources=CSV_MONTADO);

skill skill_01UQZpmDAvwa7r5DMZ5mKvK4


sesión sesn_01PKzhkXuF8GEDUJWP2K2vpW


→ read: {'file_path': '/workspace/skills/margen-por-proyecto/SKILL.md'}
→ bash: {'command': 'ls -la /mnt/session/uploads/ && head -20 /mnt/session/uploads/contabilidad.cs


→ bash: {'command': 'cd /mnt/session/uploads && uv run --with pandas python3 - <<\'PY\'\nimport pa


→ bash: {'command': 'cd /mnt/session/uploads && uv run --with pandas python3 - <<\'PY\'\nimport pa


El proyecto con la caída más fuerte es el **1045**: margen bajó de **33,7% a 11,5% (-22,2 pp)**, pese a que el ingreso subió (222,7M → 230M).

**Causa:** los costos crecieron mucho más que el ingreso, principalmente por:
- Costo de mercancía vendida: +57,5M
- Sueldos y salarios: +19,0M
- Aportes pensión/EPS: +1,6M en conjunto

(Arriendo y mantenimiento de vehículos bajaron, pero no compensaron el resto).

**Resumen general jun→jul:**
| Proyecto | Cambio (pp) |
|---|---|
| 1045 | -22,2 (mayor caída) |
| 2235 | -8,0 |
| 3310 | -3,2 |
| 4410 | -2,1 |
| 3322 | -0,3 |
| 1030 | +2,4 |
| 2210 | +10,0 (mayor mejora) |

Proyectos 0 y 5001 no tienen margen calculable (sin ingreso o ingreso cero).



https://platform.claude.com/workspaces/default/sessions/sesn_01PKzhkXuF8GEDUJWP2K2vpW


## 5 · MVP: tool propia del cliente
Una `custom` tool: el agente la pide, este notebook la ejecuta (`requires_action` → `user.custom_tool_result`).

In [6]:
(WS / "out").mkdir(exist_ok=True); (WS / "out/hallazgos-managed.jsonl").unlink(missing_ok=True)

def guardar_hallazgo(proyecto, cambio_pp, causa):
    with open(WS / "out/hallazgos-managed.jsonl", "a") as f:
        f.write(json.dumps({"proyecto": proyecto, "cambio_pp": cambio_pp, "causa": causa}, ensure_ascii=False) + "\n")
    return f"guardado {proyecto}"

a5 = agente("mvp", system=SYSTEM_CORTO + " Por cada proyecto con |cambio| > 5 pp llama guardar_hallazgo una vez.",
            tools=[{"type": "agent_toolset_20260401"},
                   {"type": "custom", "name": "guardar_hallazgo",
                    "description": "Guarda un hallazgo en el sistema del cliente: proyecto, cambio de margen en puntos porcentuales y causa principal.",
                    "input_schema": {"type": "object", "properties": {
                        "proyecto": {"type": "string"}, "cambio_pp": {"type": "number"}, "causa": {"type": "string"}},
                        "required": ["proyecto", "cambio_pp", "causa"]}}],
            skills=[{"type": "custom", "skill_id": skill.id, "version": "latest"}])
correr(a5, resources=CSV_MONTADO, herramientas={"guardar_hallazgo": guardar_hallazgo})
print("\nhallazgos-managed.jsonl:\n" + (WS / "out/hallazgos-managed.jsonl").read_text())

sesión sesn_01Akbjc8x5pbhvd8fGrqi4Qt


→ read: {'file_path': '/workspace/skills/margen-por-proyecto/SKILL.md'}
→ bash: {'command': 'ls /mnt/session/uploads/ && head -20 /mnt/session/uploads/contabilidad.csv'}


→ bash: {'command': 'cd /mnt/session/uploads && uv run --with pandas python3 - <<\'PY\'\nimport pa


→ bash: {'command': 'cd /mnt/session/uploads && uv run --with pandas python3 - <<\'PY\'\nimport pa


→ bash: {'command': 'cd /mnt/session/uploads && uv run --with pandas python3 - <<\'PY\'\nimport pa


→ bash: {'command': 'cd /mnt/session/uploads && uv run --with pandas python3 - <<\'PY\'\nimport pa


Ya tengo todo. Guardo los hallazgos y respondo.
→ custom guardar_hallazgo: {'cambio_pp': -22.171, 'causa': 'Costo de mercancía vendida casi se duplicó (+57.5M) y sueldos y salarios casi se duplicaron (+19M, de 22.3M a 41.4M), pese a que el ingreso solo creció 3%.', 'proyecto': '1045'}
→ custom guardar_hallazgo: {'cambio_pp': -8.014, 'causa': 'Mantenimiento de vehículos subió fuertemente (+14.8M, de 30.8M a 45.6M) mientras el ingreso creció solo 2.5%.', 'proyecto': '2235'}
→ custom guardar_hallazgo: {'cambio_pp': 10, 'causa': 'El ingreso creció 48.6% (de 175M a 260M), superando el aumento de costos (mantenimiento de vehículos subió 25.9M y combustibles 1M), lo que mejoró el margen.', 'proyecto': '2210'}


**Cambios de margen jun→jul (proyectos con |cambio| > 5 pp):**

| Proyecto | Jun | Jul | Cambio | Causa principal |
|---|---|---|---|---|
| 1045 | 33.7% | 11.5% | **-22.2 pp** | Costo de mercancía vendida casi se duplicó (+57.5M) y nómina casi se duplicó (+19M), con ingreso casi plano (+3%). |
| 2235 | 30.0% | 22.0% | **-8.0 pp** | Mantenimiento de vehículos subió +14.8M mientras el ingreso solo creció 2.5%. |
| 2210 | 38.0% | 48.0% | **+10.0 pp** | El ingreso creció 48.6% (175M→260M), superando el alza en mantenimiento de vehículos (+25.9M), mejorando el margen. |

Los demás proyectos (3310, 4410, 3322, 1030) tuvieron variaciones menores a 5 pp y no se registraron.



https://platform.claude.com/workspaces/default/sessions/sesn_01Akbjc8x5pbhvd8fGrqi4Qt

hallazgos-managed.jsonl:
{"proyecto": "1045", "cambio_pp": -22.171, "causa": "Costo de mercancía vendida casi se duplicó (+57.5M) y sueldos y salarios casi se duplicaron (+19M, de 22.3M a 41.4M), pese a que el ingreso solo creció 3%."}
{"proyecto": "2235", "cambio_pp": -8.014, "causa": "Mantenimiento de vehículos subió fuertemente (+14.8M, de 30.8M a 45.6M) mientras el ingreso creció solo 2.5%."}
{"proyecto": "2210", "cambio_pp": 10, "causa": "El ingreso creció 48.6% (de 175M a 260M), superando el aumento de costos (mantenimiento de vehículos subió 25.9M y combustibles 1M), lo que mejoró el margen."}
{"proyecto": "2235", "cambio_pp": -8.014, "causa": "Mantenimiento de vehículos subió fuertemente (+14.8M, de 30.8M a 45.6M) mientras el ingreso creció solo 2.5%."}
{"proyecto": "2210", "cambio_pp": 10, "causa": "El ingreso creció 48.6% (de 175M a 260M), superando el aumento de costos (mantenimiento de v

## Limpieza
Los agentes quedan versionados en el workspace; se archivan para no acumular.

In [7]:
for aid in creados:
    client.beta.agents.archive(aid)
print(f"{len(creados)} agentes archivados")

5 agentes archivados


Mismas palancas: system prompt · tools (subconjunto o todo) · archivos montados · skills · tools propias. Nada de loop, sandbox ni servidor: lo opera Anthropic.